<center>

# [Компьютерное зрение](http://rairi.ru/wiki/index.php/%D0%9A%D0%BE%D0%BC%D0%BF%D1%8C%D1%8E%D1%82%D0%B5%D1%80%D0%BD%D0%BE%D0%B5_%D0%B7%D1%80%D0%B5%D0%BD%D0%B8%D0%B5)

## <center> Семинар 11 - Трекинг

<a target="_blank" href="https://colab.research.google.com/github/alexmelekhin/cv_course_2023/blob/main/seminars/seminar_11/Seminar_11.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

***

In [ ]:
from typing import Tuple
from time import time
import types
from pathlib import Path

from scipy.optimize import linear_sum_assignment
import numpy as np
import cv2
import matplotlib.pyplot as plt

from ultralytics import YOLO

# Вспомогательные функции

In [ ]:
def show_image(img: np.ndarray) -> None:
    plt.figure(figsize=(10,5))
    plt.imshow(img)
    plt.xticks([])
    plt.yticks([])
    plt.show()


def detection_visualization(
    img_path: Path, preds: np.ndarray
) -> np.ndarray:
    img = cv2.imread(str(img_path))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    for i in range(len(preds)):
        cv2.rectangle(
            img,
            (int(preds[i][0]), int(preds[i][1])),
            (int(preds[i][2]), int(preds[i][3])),
            color=(0, 255, 0),
            thickness=3,
        )
    return img


def tracking_visualization(
    img_path: Path, preds: np.ndarray
) -> np.ndarray:
    img = cv2.imread(str(img_path))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    for i in range(len(preds)):
        cv2.rectangle(
            img,
            (int(preds[i][0]), int(preds[i][1])),
            (int(preds[i][2]), int(preds[i][3])),
            color=(0, 255, 0),
            thickness=3,
        )
        cv2.putText(
            img,
            str(int(preds[i][4])),
            (int(preds[i][0]), int(preds[i][1])),
            cv2.FONT_HERSHEY_SIMPLEX,
            fontScale=2,
            color=(0, 255, 0),
            thickness=3
        )
    return img


# Датасет

В рамках данного семинара мы будем использовать трек `MOT17-09` из датасета MOT Challenge https://motchallenge.net/

Ссылка на скачивание датасета https://motchallenge.net/data/MOT17Det/

In [ ]:
dataset_dir = Path("./Datasets/MOT17_09_imgs")
frames_list = sorted([f for f in dataset_dir.iterdir()])

In [ ]:
img = cv2.cvtColor(cv2.imread(str(frames_list[0])), cv2.COLOR_BGR2RGB)
show_image(img)

# Детекция с помощью YOLOv8

В качестве бэкбона детекции мы будем использовать YOLOv8s из библиотеки ultralytics: https://github.com/ultralytics/ultralytics

In [ ]:
class YoloDetector:
    def __init__(self, conf_threshold: float = 0.5) -> None:
        self.backbone = YOLO("yolov8s.pt")
        self.conf_threshold = conf_threshold

    def __call__(self, img_path: Path) -> np.ndarray:
        preds = self.backbone.predict(source=img_path, conf=self.conf_threshold, show=False, classes=0)
        preds = preds[0].boxes.data.cpu().numpy()[:, :5]
        return preds

In [ ]:
detector = YoloDetector()
preds = detector(frames_list[0])

out_frame = detection_visualization(frames_list[0], preds)
show_image(out_frame)

# Алгоритм трекинга SORT

SORT: https://arxiv.org/abs/1602.00763

Разделы "3.3. Data Association" и "3.4. Creation and Deletion of Track Identities" описывают алгоритм соотнесения результатов детекции последовательных кадров:

> **3.3. Data Association**
> 
> In assigning detections to existing targets, each target’s
> bounding box geometry is estimated by predicting its new
> location in the current frame. The assignment cost matrix is
> then computed as the intersection-over-union (IOU) distance
> between each detection and all predicted bounding boxes
> from the existing targets. The assignment is solved optimally
> using the Hungarian algorithm. Additionally, a minimum
> IOU is imposed to reject assignments where the detection to
> target overlap is less than $IOU_{min}$.
>
> We found that the IOU distance of the bounding boxes
> implicitly handles short term occlusion caused by passing targets.
> Specifically, when a target is covered by an occluding
> object, only the occluder is detected, since the IOU distance
> appropriately favours detections with similar scale. This allows
> both the occluder target to be corrected with the detection while 
> the covered target is unaffected as no assignment is made.

> **3.4. Creation and Deletion of Track Identities**
>
> When objects enter and leave the image, unique identities
> need to be created or destroyed accordingly. For creating
> trackers, we consider any detection with an overlap less than
> $IOU_{min}$ to signify the existence of an untracked object. The
> tracker is initialised using the geometry of the bounding box
> with the velocity set to zero. Since the velocity is unobserved
> at this point the covariance of the velocity component is initialised
> with large values, reflecting this uncertainty. Additionally, the new
> tracker then undergoes a probationary period where the target needs
> to be associated with detections to accumulate enough evidence in order
> to prevent tracking of false positives.
> 
> Tracks are terminated if they are not detected for $T_{Lost}$
> frames. This prevents an unbounded growth in the number
> of trackers and localisation errors caused by predictions over
> long durations without corrections from the detector. In all
> experiments $T_{Lost}$ is set to 1 for two reasons. Firstly, the constant
> velocity model is a poor predictor of the true dynamics
> and secondly we are primarily concerned with frame-to-frame
> tracking where object re-identification is beyond the scope of
> this work. Additionally, early deletion of lost targets aids efficiency.
> Should an object reappear, tracking will implicitly resume under a new identity.

# Венгерский алгоритм (Hungarian algorithm)

- [Wikipedia](https://en.wikipedia.org/wiki/Hungarian_algorithm)
- Реализация в библиотеке scipy: [`scipy.optimize.linear_sum_assignment`](https://docs.scipy.org/doc/scipy/reference/generated/scipy.optimize.linear_sum_assignment.html)
- Реализация в библиотеке [lap](https://github.com/gatagat/lap): `lap.lapjv` 

## Задание 1:

Реализуйте Венгерский алгоритм. Сравните возвращаемые значения с библиотечной функцией [`scipy.optimize.linear_sum_assignment`](https://docs.scipy.org/doc/scipy/reference/generated/scipy.optimize.linear_sum_assignment.html).

In [ ]:
def hungarian_algorithm(cost_matrix: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
    cost = np.asarray(cost_matrix, dtype=float)
    if cost.ndim != 2:
        raise ValueError("cost_matrix must be a 2D array")

    n_rows, n_cols = cost.shape
    if n_rows == 0 or n_cols == 0:
        return np.array([], dtype=int), np.array([], dtype=int)

    transposed = n_rows > n_cols
    if transposed:
        cost = cost.T
        n_rows, n_cols = cost.shape

    u = np.zeros(n_rows + 1)
    v = np.zeros(n_cols + 1)
    p = np.zeros(n_cols + 1, dtype=int)
    way = np.zeros(n_cols + 1, dtype=int)

    for i in range(1, n_rows + 1):
        p[0] = i
        j0 = 0
        minv = np.full(n_cols + 1, np.inf)
        used = np.zeros(n_cols + 1, dtype=bool)

        while True:
            used[j0] = True
            i0 = p[j0]
            delta = np.inf
            j1 = 0

            for j in range(1, n_cols + 1):
                if used[j]:
                    continue
                cur = cost[i0 - 1, j - 1] - u[i0] - v[j]
                if cur < minv[j]:
                    minv[j] = cur
                    way[j] = j0
                if minv[j] < delta:
                    delta = minv[j]
                    j1 = j

            for j in range(n_cols + 1):
                if used[j]:
                    u[p[j]] += delta
                    v[j] -= delta
                else:
                    minv[j] -= delta

            j0 = j1
            if p[j0] == 0:
                break

        while True:
            j1 = way[j0]
            p[j0] = p[j1]
            j0 = j1
            if j0 == 0:
                break

    row_ind = []
    col_ind = []
    for j in range(1, n_cols + 1):
        if p[j] != 0:
            row_ind.append(p[j] - 1)
            col_ind.append(j - 1)

    row_ind = np.asarray(row_ind, dtype=int)
    col_ind = np.asarray(col_ind, dtype=int)

    if transposed:
        row_ind, col_ind = col_ind, row_ind

    order = np.argsort(row_ind)
    return row_ind[order], col_ind[order]


In [ ]:
cost_matrix = np.array([[1, 2, 3],
                        [4, 5, 6],
                        [7, 8, 9]])
my_row_ind, my_col_ind = hungarian_algorithm(cost_matrix)
my_cost = cost_matrix[my_row_ind, my_col_ind].sum()
row_ind, col_ind = linear_sum_assignment(cost_matrix)
cost = cost_matrix[row_ind, col_ind].sum()

assert my_cost == cost

# Алгоритм трекинга ByteTrack

- Статья https://arxiv.org/abs/2110.06864
- Код https://github.com/ifzhang/ByteTrack


## Вопрос 1:

В чем заключается ключевая особенность метода BYTE?

**Ответ:** ключевая идея BYTE заключается в том, что алгоритм не отбрасывает все низкоуверенные детекции сразу. Сначала он сопоставляет треки с высокоуверенными детекциями, а затем использует оставшиеся низкоуверенные детекции для повторного сопоставления потерянных треков. Это помогает сохранять ID при частичных перекрытиях и временном падении confidence у детектора, при этом фоновые ложные срабатывания обычно не закрепляются за треками.

In [ ]:
# %cd ByteTrack
# %cd ..


In [ ]:
from yolox.tracker.byte_tracker import BYTETracker

fps = 30
frame_size = (1080, 1920)
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
output_file = 'output.mp4'
video_writer = cv2.VideoWriter(output_file, fourcc, fps, frame_size[::-1])

args = types.SimpleNamespace(**{
    "track_thresh": 0.5,
    "track_buffer": 30,
    "match_thresh": 0.8,
    "mot20": False,

})
tracker = BYTETracker(args, frame_rate=fps)

det_times = []
track_times = []

for frame_path in frames_list:
    s_time = time()
    detections = detector(frame_path)
    det_times.append((time() - s_time) * 1000)
    s_time = time()
    tracks = tracker.update(detections, frame_size, frame_size)
    track_times.append((time() - s_time) * 1000)
    tracks = [t.tlbr.tolist() + [t.track_id] for t in tracks]
    out_frame = tracking_visualization(frame_path, tracks)
    video_writer.write(cv2.cvtColor(out_frame, cv2.COLOR_RGB2BGR))
video_writer.release()

print(f"mean detection time: {np.mean(det_times)}, mean track time: {np.mean(track_times)}")

## Задание 2

Объедините BYTE и любой трекер с Re-ID признаками (например, FairMOT). Можно воспользоваться инструкциями из репозитория: https://github.com/ifzhang/ByteTrack/tree/main/tutorials

Ответом на это задание должен быть код, записывающий видео работы метода для трека MOT17-09 в файл `reid_out.mp4`. Убедитесь, что код работает "end-to-end". 

In [ ]:
from yolox.tracker.byte_tracker import BYTETracker


def _clip_box(box: np.ndarray, width: int, height: int) -> Tuple[int, int, int, int]:
    x1, y1, x2, y2 = np.asarray(box[:4], dtype=float)
    x1 = int(np.clip(round(x1), 0, width - 1))
    y1 = int(np.clip(round(y1), 0, height - 1))
    x2 = int(np.clip(round(x2), x1 + 1, width))
    y2 = int(np.clip(round(y2), y1 + 1, height))
    return x1, y1, x2, y2


def extract_reid_feature(frame_rgb: np.ndarray, box: np.ndarray) -> np.ndarray:
    height, width = frame_rgb.shape[:2]
    x1, y1, x2, y2 = _clip_box(box, width, height)
    crop = frame_rgb[y1:y2, x1:x2]
    if crop.size == 0:
        return np.zeros(16 * 16, dtype=np.float32)

    crop = cv2.resize(crop, (64, 128), interpolation=cv2.INTER_LINEAR)
    hsv = cv2.cvtColor(crop, cv2.COLOR_RGB2HSV)
    hist = cv2.calcHist([hsv], [0, 1], None, [16, 16], [0, 180, 0, 256]).astype(np.float32)
    hist = hist.flatten()
    norm = np.linalg.norm(hist)
    return hist / norm if norm > 0 else hist


def cosine_similarity(a: np.ndarray, b: np.ndarray) -> float:
    denom = np.linalg.norm(a) * np.linalg.norm(b)
    return float(np.dot(a, b) / denom) if denom > 0 else 0.0


def center_distance(box_a: np.ndarray, box_b: np.ndarray) -> float:
    ax = (box_a[0] + box_a[2]) / 2
    ay = (box_a[1] + box_a[3]) / 2
    bx = (box_b[0] + box_b[2]) / 2
    by = (box_b[1] + box_b[3]) / 2
    return float(np.hypot(ax - bx, ay - by))


class ByteReIDTracker:
    def __init__(
        self,
        args: types.SimpleNamespace,
        frame_rate: int,
        reid_threshold: float = 0.82,
        max_reid_age: int = 90,
        feature_momentum: float = 0.9,
    ) -> None:
        self.byte_tracker = BYTETracker(args, frame_rate=frame_rate)
        self.reid_threshold = reid_threshold
        self.max_reid_age = max_reid_age
        self.feature_momentum = feature_momentum
        self.next_reid_id = 1
        self.byte_to_reid = {}
        self.gallery = {}

    def _new_reid_id(self) -> int:
        reid_id = self.next_reid_id
        self.next_reid_id += 1
        return reid_id

    def _match_gallery(self, feature: np.ndarray, box: np.ndarray, used_ids: set, frame_diag: float):
        best_id = None
        best_score = -np.inf
        for reid_id, item in self.gallery.items():
            if reid_id in used_ids or item["age"] > self.max_reid_age:
                continue
            distance_gate = center_distance(box, item["box"]) < 0.25 * frame_diag
            if not distance_gate:
                continue
            score = cosine_similarity(feature, item["feature"])
            if score > best_score:
                best_score = score
                best_id = reid_id
        return best_id if best_score >= self.reid_threshold else None

    def update(self, detections: np.ndarray, frame_rgb: np.ndarray, frame_size: Tuple[int, int]):
        for item in self.gallery.values():
            item["age"] += 1

        online_tracks = self.byte_tracker.update(detections, frame_size, frame_size)
        frame_diag = float(np.hypot(frame_size[0], frame_size[1]))
        used_reid_ids = set()
        outputs = []

        for track in online_tracks:
            byte_id = track.track_id
            box = np.asarray(track.tlbr, dtype=float)
            feature = extract_reid_feature(frame_rgb, box)

            if byte_id in self.byte_to_reid:
                reid_id = self.byte_to_reid[byte_id]
            else:
                reid_id = self._match_gallery(feature, box, used_reid_ids, frame_diag)
                if reid_id is None:
                    reid_id = self._new_reid_id()
                self.byte_to_reid[byte_id] = reid_id

            old_feature = self.gallery.get(reid_id, {}).get("feature", feature)
            updated_feature = self.feature_momentum * old_feature + (1 - self.feature_momentum) * feature
            norm = np.linalg.norm(updated_feature)
            if norm > 0:
                updated_feature = updated_feature / norm

            self.gallery[reid_id] = {"feature": updated_feature, "box": box, "age": 0}
            used_reid_ids.add(reid_id)
            outputs.append(box.tolist() + [reid_id])

        return outputs


if len(frames_list) == 0:
    raise ValueError(f"No frames found in {dataset_dir}")

first_frame = cv2.imread(str(frames_list[0]))
if first_frame is None:
    raise ValueError(f"Could not read first frame: {frames_list[0]}")

fps = 30
height, width = first_frame.shape[:2]
frame_size = (height, width)
fourcc = cv2.VideoWriter_fourcc(*"mp4v")
output_file = "reid_out.mp4"
video_writer = cv2.VideoWriter(output_file, fourcc, fps, (width, height))

args = types.SimpleNamespace(**{
    "track_thresh": 0.5,
    "track_buffer": 30,
    "match_thresh": 0.8,
    "mot20": False,
})
tracker = ByteReIDTracker(args, frame_rate=fps)

det_times = []
track_times = []

for frame_path in frames_list:
    frame_bgr = cv2.imread(str(frame_path))
    if frame_bgr is None:
        continue
    frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)

    s_time = time()
    detections = detector(frame_path)
    det_times.append((time() - s_time) * 1000)

    s_time = time()
    tracks = tracker.update(detections, frame_rgb, frame_size)
    track_times.append((time() - s_time) * 1000)

    out_frame = tracking_visualization(frame_path, tracks)
    video_writer.write(cv2.cvtColor(out_frame, cv2.COLOR_RGB2BGR))

video_writer.release()

print(f"wrote {output_file}")
print(f"mean detection time: {np.mean(det_times):.2f} ms, mean BYTE+ReID track time: {np.mean(track_times):.2f} ms")
